In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [5]:
print(documents[0])

{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}


In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [9]:
question = "How will I get a certificate?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'

In [10]:
[doc["question"] for doc in search_results]

['Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I missed the first homework - can I still get a certificate?',
 'Will the name I put in the certificate field be shown publicly online or shared with third parties?',
 'Does the course certificate show the number of course hours?',
 'I submitted my capstone project. What are the remaining certificate requirements, and when do I get my score and feedback?']

In [11]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
question="How long is the course?"
search_results = search(question)
print(search_results)

[{'id': '418e8948e7', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: How do I start?', 'answer': 'No matter if you\'re with a \'live\' cohort or following in self-paced mode, start by:\n\n- Reading pins and bookmarks on the course channel to see what things are where.\n\n  <{IMAGE:image_1}>\n\n  <{IMAGE:image_2}>\n\n- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).\n\n- If you have questions, search the channel itself first; someone may have already asked and gotten a solution.\n\n- For the most Frequently Asked Questions, refer to this document:\n\n  <{IMAGE:image_3}>\n\n- If you don\'t want to read/skimmer/search the FAQ document, tag the `@ZoomcampQABot` when asking questions, and it will summarize answers from its knowledge base.\n\n- For generic, non-zoomcamp queries, consider using tools like ChatGPT, BingCopilot, or Google Gemini, especially for error messages.\n\n- C

In [15]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [17]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [19]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [20]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
How long is the course?

Context:
General Course-Related Questions
Q: Course: How do I start?
A: No matter if you're with a 'live' cohort or following in self-paced mode, start by:

- Reading pins and bookmarks on the course channel to see what things are where.

  <{IMAGE:image_1}>

  <{IMAGE:image_2}>

- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).

- If you have questions, search the channel itself first; someone may have already asked and gotten a solution.

- For the most Frequently Asked Questions, refer to this document:

  <{IMAGE:image_3}>

- If you don't want to read/skimmer/search the FAQ document, tag the `@ZoomcampQABot` when asking questions, and it will summarize answers from its knowledge base.

- For generic, non-zoomcamp queries, consider using tools like ChatGPT, BingCopilot, or Google Gemini, especially for error messages.

- Check if you're on track by checking the deadlines in the Course 

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    print(response.usage)
    return response.output_text

...

Ellipsis

In [27]:
llm(prompt)


'The course length isn’t specified here, but it has **homework deadlines** and a **final project deadline**, so it runs over a set period rather than being open-ended.\n\nIf you want, I can also help you find the exact course duration from the course schedule or FAQ.'